# AGOL Demo | Load One Module File from GitHub

This notebook shows a simple pattern for loading one Python module file directly from a GitHub repository into an AGOL notebook session.

**What the helper function `load_github_module` does:**

1. Builds a raw GitHub URL for one `.py` file.
2. Downloads the file to a local cache in `/arcgis/home/modules_cache`.
3. Imports the file dynamically as a Python module.
4. Returns the imported module so you can call its functions.

*Why use this approach?*

- You can use one module file without installing the full package.
- You can pin `reference` to a tag or commit for reproducible scheduled runs.
- You can use `main` for fast development iteration.

In [11]:
import os
import urllib.request
import importlib.util

def load_github_module(
    module: str,
    owner: str,
    repo: str,
    reference: str | None = None,
):
    """
    Load one Python module file from a GitHub repository into the current AGOL session.

    `reference` can be a branch, tag, or commit hash.

    :param module: Path to module file in repository, for example "modules/hello_world.py".
    :param owner: GitHub owner or organization.
    :param repo: GitHub repository name.
    :param reference: Branch, tag, or commit hash. Defaults to "main".
    :return: Imported module object.
    """
    reference = reference or "main"

    local_file = f"/arcgis/home/modules_cache/{reference[:7]}/{module}"
    os.makedirs(os.path.dirname(local_file), exist_ok=True)

    url = f"https://raw.githubusercontent.com/{owner}/{repo}/{reference}/{module}"
    if not os.path.exists(local_file):
        urllib.request.urlretrieve(url, local_file)

    module_stub = module.removesuffix(".py")
    module_name = f"{module_stub}_{reference[:7]}"

    spec = importlib.util.spec_from_file_location(module_name, local_file)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not create import spec for {local_file}")

    loaded_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(loaded_module)
    return loaded_module


In [12]:
# Example A: load module `hello.py` by version tag (reproducible, recommended for stable releases)
hello = load_github_module(
    owner="miljodirektoratet",
    repo="arcpy-utils-public",
    reference="v0.0.1",
    module="src/mdir_arcpy_utils_public/hello.py",
)
hello.main()

Hello from mdir-arcpy-utils-public!
Version: 0.0.1


In [9]:
# Example B: load module `hello.py` by commit hash (reproducible)
hello = load_github_module(
    owner="miljodirektoratet",
    repo="arcpy-utils-public",
    reference="79890ee810d0f333393af8e4407cb4f6573f6532",
    module="src/mdir_arcpy_utils_public/hello.py",
)
hello.main()

Hello from mdir-arcpy-utils-public!
Version: 0.0.1


In [10]:
# Example C: load module `hello.py` by branch name (not reproducible, only recommended for active development not for notebook-jobs)
hello = load_github_module(
    owner="miljodirektoratet",
    repo="arcpy-utils-public",
    reference="main",
    module="src/mdir_arcpy_utils_public/hello.py",
)

# load main function from module `src/mdir_arcpy_utils_public/hello.py`
hello.main()

Hello from mdir-arcpy-utils-public!
Version: 0.0.1


In [28]:
# Example D: load the package `mdir_arcpy_utils_public` (not recommended, costs more credits)
%pip install --no-cache-dir "https://github.com/miljodirektoratet/arcpy-utils-public/archive/refs/tags/v0.0.3.zip"

     - 15.3 kB 103.7 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mdir-arcpy-utils-public: filename=mdir_arcpy_utils_public-0.0.3-py3-none-any.whl size=4072 sha256=ae72ebaee0423cc1a8e29812b5bfe0e73181fdb7637a4d0586c2bb4ddc062348
  Stored in directory: /tmp/pip-ephem-wheel-cache-85nd_g2t/wheels/86/f1/09/366539b8085b6e4b27b1630b4385a25967a101654d11cee450
Successfully built mdir-arcpy-utils-public
  Attempting uninstall: mdir-arcpy-utils-public
    Found existing installation: mdir-arcpy-utils-public 0.0.1
    Uninstalling mdir-arcpy-utils-public-0.0.1:
      Successfully uninstalled mdir-arcpy-utils-public-0.0.1
Note: you may need to restart the kernel to use updated packages.


In [29]:
import mdir_arcpy_utils_public as mdir_arcpy
mdir_arcpy.main()

Hello from mdir-arcpy-utils-public!
Version: 0.0.3


In [30]:
from mdir_arcpy_utils_public import hello
hello.main()

Hello from mdir-arcpy-utils-public!
Version: 0.0.3
